# Ứng dụng dự đoán điểm thi (Demo)
> **Cảnh báo:** Ứng dụng và mô hình sử dụng dữ liệu tổng hợp cho mục đích học tập/giảng dạy, không sử dụng để đánh giá sinh viên thực tế.

In [ ]:
from pathlib import Path
import joblib
import numpy as np 
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

MODEL_PATH = Path("artifacts/linear_regression.joblib")
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

artifact = joblib.load(MODEL_PATH)
model = artifact["model"]
feature_names = artifact["feature_names"]

In [ ]:
hours = widgets.FloatSlider(value=6.0, min=0.0, max=20.0, step=0.5, description="Giờ học:", continuous_update=False)
attendance = widgets.FloatSlider(value=80.0, min=50.0, max=100.0, step=1.0, description="Chuyên cần:", continuous_update=False)
assignments = widgets.IntSlider(value=7, min=0, max=10, step=1, description="Bài tập:", continuous_update=False)
previous = widgets.FloatSlider(value=6.5, min=0.0, max=10.0, step=0.1, description="Điểm trước:", continuous_update=False)

predict_button = widgets.Button(description="Dự đoán", button_style="primary")
output = widgets.Output()

def input_frame():
    values = {
        "hours_studied": hours.value,
        "attendance": attendance.value,
        "assignments_completed": assignments.value,
        "previous_score": previous.value,
    }
    return pd.DataFrame([[values[name] for name in feature_names]], columns=feature_names)

def on_predict(_):
    predict_button.disabled = True
    with output:
        output.clear_output()
        try:
            raw = float(model.predict(input_frame())[0])
            pred_clamped = np.clip(raw, 0, 10)
            print(f"Điểm thi dự đoán: {pred_clamped:.2f} / 10")
        except Exception:
            print("Không thể dự đoán. Hãy kiểm tra dữ liệu và artifact.")
        finally:
            predict_button.disabled = False

predict_button.on_click(on_predict)
display(widgets.VBox([hours, attendance, assignments, previous, predict_button, output]))